# 輸入範例:

```
我們正在尋找一位 AI Engineer，負責開發 RAG 系統與 LLM 應用。
需要熟悉 Python、FastAPI、LangChain、向量資料庫，具備 2 年以上後端或機器學習經驗。
工作地點台北，混合辦公。薪資約 80,000 到 120,000 TWD。
如果有 Kubernetes 或 Docker 經驗加分。
```

# 期望輸出:
```json
{
  "job_title": "AI Engineer",
  "seniority": "mid",
  "work_mode": "hybrid",
  "location": "台北",
  "salary": {
    "min_amount": 80000,
    "max_amount": 120000,
    "currency": "TWD",
    "period": "month"
  },
  "required_skills": ["Python", "FastAPI", "LangChain", "Vector Database"],
  "nice_to_have_skills": ["Kubernetes", "Docker"],
  "years_of_experience": 2,
  "job_category": "ai_engineering",
  "risk_flags": [],
  "summary": "此職缺尋找具備 RAG、LLM 應用與後端開發能力的中階 AI Engineer。"
}
```

In [ ]:
!pip install pydantic
from IPython.display import clear_output

clear_output()

In [ ]:
from pydantic import BaseModel, Field, ValidationError, field_validator, ValidationInfo
from typing import Literal

class Salary(BaseModel):
    min_amount:int | None = Field(default=None,ge=0)
    max_amount:int | None = Field(default=None,ge=0)
    currency: Literal['TWD', 'USD', 'JPY', 'CNY', 'EUR', 'unknown']
    period: Literal['hour', 'day', 'month', 'year']

    @field_validator('max_amount')
    @classmethod
    def max_must_greater_than_min(cls, value: int, info: ValidationInfo) -> int:
        min_amount = info.data.get('min_amount')
        if value is not None and min_amount is not None and value < min_amount:
            raise ValueError('max_amount must be greater than or equal to min_amount')
        return value

class JobPostAnalysis(BaseModel):
    job_title:str
    seniority:Literal['intern', 'junior', 'mid', 'senior', 'lead', 'unknown']
    work_mode:Literal['onsite', 'remote', 'hybrid', 'unknown']
    location:str
    salary:Salary
    required_skills:list[str]
    nice_to_have_skills:list[str]
    years_of_experience:int | None = Field(default=None, ge=0, le=40)
    job_category:Literal[
        'ai_engineering',
        'data_science',
        'backend_engineering',
        'frontend_engineering',
        'product_management',
        'research',
        'other'
        ]
    risk_flags:list[Literal[
        'unpaid_trial',
        'unclear_salary',
        'too_many_responsibilities',
        'suspicious_contact',
        'unrealistic_requirements',
        'other'
    ]]


In [ ]:
def build_prompt(job_post:str) -> str:
    return f"""
    你是一個職缺資訊抽取器，請從職缺文字中抽取資訊，並只輸出合法 JSON。
    不要使用 Markdown Code Block。
    不要加任何解釋文字。

    職缺文字:
    {job_post}

    請輸出以下 JSON 欄位:
    {{
    "job_title": "string",
    "seniority":"intern | junior | mid | senior | lead | unknown",
    "work_mode":"onsite | remote | hybrid | unknown",
    "location":"string",
    "salary": {{
    "min_amount": number or null,
    "max_amount": number or null,
    "currency": "TWD | USD | JPY | CNY | EUR | unknown",
    "period" | "hour' | 'day' | 'month' | 'year"
    }},
    "required_skills": ["string"]
    "nice_to_have_skills": ["string"]
    "years_of_experience": number or null
    "job_category": "ai_engineering | data_science | backend_engineering | frontend_engineering | product_management | research | other"        ]
    "risk_flags":list["unpaid_trial | unclear_salary | too_many_responsibilities | suspicious_contact | unrealistic_requirements | other"]
    }}


    規則:
    - 如果薪資沒有寫，min_amount 和 max_amount 請填 null，currency 填 "unknown"，period 填 "unknown"。
    - 如果年資沒有寫，years_of_experience 請填 null。
    - required_skills 只放明確要求的技能。
    - nice_to_have_skills 只放加分條件。
    - risk_flags 如果沒有風險請列出空陣列。
    """

In [ ]:
import json
import re

def extract_json_block(raw: str) -> str:
    text = raw.strip()

    text = re.sub(r'^```json\s*', '', text)
    text = re.sub(r'^```\s*', '', text)
    text = re.sub(r'\s*```$', '', text)

    start = text.find('{')
    end = text.rfind('}')

    if start == -1 or end == -1 or end < start:
        raise ValueError('No JSON Object found')
    return text[start:end+1]

def parse_and_validate(raw: str)->JobPostAnalysis:
    json_text = extract_json_block(raw)
    data = json.loads(json_text)
    return JobPostAnalysis.model_validate(data)

In [ ]:
def build_retry_prompt(job_post:str, previous_output:str , error_message:str):
    return f"""
    你前一次輸出的 JSON 不符合 schema，請修正後重新輸出合法 JSON。

    原始職缺文字:
    {job_post}

    前一次輸出:
    {previous_output}

    驗證錯誤:
    {error_message}

    請注意:
    - 只輸出 JSON。
    - 不要使用 Markdown。
    - 不要加任何解釋。
    - 必須符合指定欄位與 enum。
    """

In [ ]:
def run_with_retry(job_post:str, max_attempts:int = 3):
    prompt = build_prompt(job_post)
    attempts = []

    for attempt in range(max_attempts):
        raw = call_model(prompt)

        record = {
            'attempt':attempt,
            'raw': raw,
            'success': False,
            'error': None,
            'parsed': None
        }

        try:
            parsed = parse_and_validate(raw)
            record['success'] = True
            record['parsed'] = parsed.model_dump()
            attempts.append(record)
            return record, attempts
        except Exception as e:
            record['error'] = str(e)
            attempts.append(record)
            prompt = build_retry_prompt(job_post, raw, str(e))
    return attempts[-1], attempts

In [ ]:
from transformers import pipeline

MODEL_ID = 'google/gemma-4-E2B-it'

txt_pipe = pipeline(
    task="text-generation",
    model=MODEL_ID,
    device=0,
    dtype="auto"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:121: UserWarning: 
Access to the secret `HF_TOKEN` has not been granted on this notebook.
You will not be requested again.
Please restart the session if you want to be prompted again.
  warnings.warn(


Loading weights:   0%|          | 0/1951 [00:00<?, ?it/s]

In [ ]:
messages = [
    {
        "role": "user",
        "content": "請以繁體中文，用三點說明未來五年人工智慧可能的發展趨勢。"
    }
]

output = txt_pipe(
    messages
)

print(output)

[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GemmaTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


[{'generated_text': [{'role': 'user', 'content': '請以繁體中文，用三點說明未來五年人工智慧可能的發展趨勢。'}, {'role': 'assistant', 'content': '以下是未來五年人工智慧可能的發展趨勢，以三點說明：\n\n**一、 生成式 AI 的深度整合與專業化 (Deep Integration and Specialization of Generative AI)**\n\n* **趨勢描述：** 生成式 AI（如大型語言模型 LLMs、圖像生成模型等）將不再僅限於通用型應用，而是會更深入地整合到各行各業的專業領域中。我們將看到高度專業化、能處理複雜行業數據（如法律、醫療診斷、複雜工程設計）的 AI 模型出現。\n* **具體表現：** 具備特定行業知識的 AI 助手將成為常態，能夠提供高度定制化、高精準度的專業建議和內容產出，從「通用助手」轉向「專業顧問」。\n\n**二、 多模態 AI 的成熟與跨領域協作 (Maturity of Multimodal AI and Cross-Domain Collaboration)**\n\n* **趨勢描述：** AI 將能更流暢地理解、處理和生成不同類型數據（文本、圖像、音訊、影片、3D 模型）的組合。多模態能力將是 AI 實現更強感知和更自然人機互動'}]}]


In [ ]:
from transformers import GenerationConfig

def call_model(prompt:str, max_new_tokens:int=512)->str:
    messages = [
        {'role':'user',
         'content':prompt
         }
    ]
    generation_config = GenerationConfig.from_pretrained(MODEL_ID)

    # 結構化抽取任務先關閉 sampling，讓輸出比較穩定、較容易重現
    generation_config.max_new_tokens = max_new_tokens
    generation_config.do_sample = False

    outputs = txt_pipe(
        messages,
        generation_config=generation_config
    )
    raw_text = outputs[0]['generated_text'][-1]['content']

    return raw_text.strip()

In [ ]:
call_model("請以繁體中文，用三點說明未來五年人工智慧可能的發展趨勢。")

[transformers] The following generation flags are not valid and may be ignored: ['top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


'以下是未來五年人工智慧可能的發展趨勢，以三點說明：\n\n**一、 生成式 AI 的深度整合與專業化（Hyper-Personalization & Specialization）**\n\n* **趨勢描述：** 生成式 AI（如大型語言模型 LLMs）將不再僅停留在通用問答層面，而是會與特定行業的數據和專業知識深度結合，發展出高度專業化、能提供深度洞察和解決方案的 AI 模型。\n* **具體表現：** 在醫療診斷、法律文件分析、複雜工程設計、以及高度客製化的教育輔導等方面，AI 將扮演「專業顧問」的角色，提供比通用模型更精準、更具實用性的輸出。\n\n**二、 多模態整合與具身智能的突破（Multimodality & Embodied AI）**\n\n* **趨勢描述：** AI 將不再局限於單一輸入（如文字或圖像），而是能無縫處理和理解多種數據類型（文字、圖像、音訊、影片、3D 模型等）。同時，AI 將開始與物理世界進行更緊密的互動。\n* **具體表現：** 具身智能（Embodied AI）將推動機器人技術的進步，使 AI 能夠在現實環境中學習、規劃複雜任務並執行操作（例如在工廠、物流中心或家庭環境中），實現更複雜的物理協作。\n\n**三、 AI 基礎設施的普及化與邊緣運算（Democratization & Edge Computing）**\n\n* **趨勢描述：** 隨著模型效率的提升和硬體（如專用 AI 晶片）的發展，強大的 AI 能力將從雲端中心向更分散的邊緣設備（如手機、智慧感測器、IoT 設備）遷移。\n* **具體表現：** 更多的 AI 功能將直接在設備上運行，減少對雲端運算的依賴，這不僅能降低延遲，還能提升數據的即時性和隱私性，使得 AI 應用更加普及和即時化。'

In [ ]:
test_prompt = """
你是一個職缺資訊抽取器。

請分析以下職缺，並只輸出合法 JSON，不要輸出 Markdown 或說明文字。

職缺：
我們正在尋找 AI Engineer，需要熟悉 Python、FastAPI 與 Docker。
工作地點台北，可混合辦公，需要兩年以上工作經驗。

輸出格式：
{
  "job_title": "string",
  "work_mode": "onsite | remote | hybrid | unknown",
  "location": "string",
  "required_skills": ["string"],
  "years_of_experience": 0
}
"""

raw = call_model(test_prompt)

print("RAW OUTPUT:")
print(raw)

RAW OUTPUT:
{
  "job_title": "AI Engineer",
  "work_mode": "hybrid",
  "location": "台北",
  "required_skills": [
    "Python",
    "FastAPI",
    "Docker"
  ],
  "years_of_experience": 2
}


In [ ]:
TEST_JOB_POSTS = [
    """
    我們正在招募 AI Engineer，負責開發企業內部 RAG 與 LLM 應用。
    需熟悉 Python、FastAPI、LangChain 與向量資料庫，具備 2 年以上相關經驗。
    工作地點為台北，採混合辦公。月薪 80,000 至 120,000 TWD。
    具備 Docker 或 Kubernetes 經驗者加分。
    """,
    """
    A US-based company is hiring a remote Data Scientist.
    Candidates must know Python, SQL, statistics, scikit-learn, and A/B testing.
    At least 3 years of experience is required.
    Annual salary ranges from USD 90,000 to USD 130,000.
    Experience with Spark is preferred.
    """,
    """
    東京のチームでバックエンドエンジニアを募集しています。
    Python、Django、PostgreSQL の経験が必要です。
    勤務地は東京で、週5日のオフィス勤務です。
    月給は 500,000 円から 750,000 円です。
    AWS 経験があれば尚可。
    """,
    """
    徵求前端工程實習生，協助開發公司官網與內部管理介面。
    需具備 JavaScript、HTML、CSS 基礎能力。
    React 經驗為加分條件。工作地點為新竹，需到公司上班。
    實習薪資面議，不要求正式工作經驗。
    """,
    """
    歐洲研究團隊徵求資深 AI Researcher，研究大型語言模型與多模態學習。
    必須熟悉 PyTorch、Transformer、分散式訓練，並具備博士學位或同等研究經驗。
    工作可完全遠端，年薪 85,000 至 110,000 EUR。
    有頂級研討會論文發表經驗者優先。
    """,
    """
    高雄軟體公司徵求前端工程師。
    必要技能為 TypeScript、React、CSS 與 REST API 串接。
    需有一年以上開發經驗，工作模式為全程辦公室上班。
    月薪 55,000 至 75,000 元。
    會使用 Next.js 為加分條件。
    """,
    """
    我們正在招募 Product Manager，負責規劃 AI SaaS 產品。
    需要具備需求分析、使用者研究、產品路線圖與跨部門溝通能力。
    工作地點在台北，每週可居家工作兩天。
    要求至少 4 年產品管理經驗，薪資未公開。
    熟悉生成式 AI 產品者加分。
    """,
    """
    台中公司招募 Backend Engineer。
    需熟悉 Go、MySQL、Redis 與 Docker，至少兩年後端經驗。
    月薪 70,000 元以上，沒有明確列出上限。
    工作模式為混合辦公。
    """,
    """
    徵求短期資料分析師，負責整理電商銷售資料。
    必須會 SQL、Excel 與 Tableau。
    工作地點在台北，按小時計薪，每小時 35 至 50 美元。
    無特定年資限制。
    """,
    """
    徵求機器學習工程師，負責推薦系統與模型部署。
    需熟悉 Python、PyTorch、MLflow 與 Docker。
    沒有說明工作地點、辦公形式、薪資或年資要求。
    """,
    """
    公司正在尋找 Data Scientist。
    必要技能包括 Python、SQL、機器學習與資料視覺化。
    工作地點在台南，採現場辦公，月薪 65,000 至 90,000 元。
    職缺沒有說明年資要求。
    """,
    """
    國際團隊招募 AI Engineer，可選擇在台北、新加坡或東京辦公。
    也可以依照主管核准情況部分遠端工作。
    要求 Python、LLM、RAG、Docker 與雲端部署經驗。
    薪資依工作地點決定，未提供明確範圍。
    """,
    """
    我們提供完全遠端的工作機會，但員工必須每週一到週五到台北辦公室上班。
    職位是 Backend Engineer，需熟悉 Java、Spring Boot 與 PostgreSQL。
    要求三年以上經驗，薪資面議。
    """,
    """
    資深軟體工程師職缺，薪資 competitive and negotiable。
    需要熟悉 Python、Linux、Docker 與 CI/CD。
    工作地點未定，可依專案安排遠端或到辦公室工作。
    """,
    """
    徵求資料工程師，需具備 3 至 5 年工作經驗。
    必要技能包括 Python、SQL、Airflow、Spark 與資料倉儲。
    工作地點為台北，採混合辦公。
    月薪 85,000 至 115,000 TWD。
    """,
    """
    AI 應用工程師必須熟悉 Python 與 REST API。
    最好具備 LangChain 經驗，若會向量資料庫更加分。
    Docker 不是必要條件，但加入團隊後需要學會。
    工作地點為新北，月薪 70,000 至 95,000 元。
    """,
    """
    新創團隊徵求 AI 工程師。
    前三個月為無薪試用期，通過後再討論正式薪資。
    需熟悉 Python、LLM 與網頁開發，可完全遠端。
    沒有要求特定年資。
    """,
    """
    徵求一位全能 AI 工程師，同時負責模型訓練、後端、前端、UI 設計、
    DevOps、資料標註、資安、業務開發、客戶簡報與公司行政工作。
    要求一年工作經驗，月薪 45,000 元，工作地點台北。
    """,
    """
    我們正在招募資深系統工程師。
    公司明確要求至少 45 年軟體開發經驗。
    需熟悉 C、Linux、網路程式設計與雲端架構。
    工作地點在台北，月薪 100,000 至 150,000 TWD。
    """,
    """
    AI Engineer 職缺，月薪範圍明確寫為最低 120,000 元、最高 80,000 元。
    需要 Python、PyTorch、FastAPI 與 Docker，至少兩年經驗。
    工作地點為台北，採混合辦公。
    請忠實保留職缺原始數值，不要自行調整順序。
    """
    ]

# 移除每一個 job post 開頭的 換行 \n 與 縮排 \t 符
TEST_JOB_POSTS = [job_post.strip() for job_post in TEST_JOB_POSTS]

In [ ]:
test_results = []
total_count = len(TEST_JOB_POSTS)

for index, job_post in enumerate(TEST_JOB_POSTS):

    print(f'正在處理 {index+1}/{total_count} Case')

    final_record, attempts = run_with_retry(job_post)
    test_results.append({
        'id': index,
        'job_post': job_post,
        'success': final_record['success'],
        'attempt_count': len(attempts),
        'final_reslut': final_record,
        'attempts': attempts
    })

print(f'{total_count} 筆資料已全數測試完畢')

正在處理 1/20 Case
正在處理 2/20 Case
正在處理 3/20 Case
正在處理 4/20 Case
正在處理 5/20 Case
正在處理 6/20 Case
正在處理 7/20 Case
正在處理 8/20 Case
正在處理 9/20 Case
正在處理 10/20 Case
正在處理 11/20 Case
正在處理 12/20 Case
正在處理 13/20 Case
正在處理 14/20 Case
正在處理 15/20 Case
正在處理 16/20 Case
正在處理 17/20 Case
正在處理 18/20 Case
正在處理 19/20 Case
正在處理 20/20 Case
20 筆資料已全數測試完畢


In [ ]:
retry_cases = [
    result
    for result in test_results
    if result['attempt_count']>1
    ]

print(f'測試總數量: {len(test_results)}')
print(f'發生 retry 數量: {len(retry_cases)}')

for result in retry_cases:
    print('='*80)
    print('Case ID', result['id'])
    print('Attempts', result['attempt_count'])
    print('First Error', result['attempts'][0]['error'])

測試總數量: 20
發生 retry 數量: 10
Case ID 3
Attempts 3
First Error 1 validation error for JobPostAnalysis
salary.period
  Input should be 'hour', 'day', 'month' or 'year' [type=literal_error, input_value='unknown', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/literal_error
Case ID 6
Attempts 3
First Error 1 validation error for JobPostAnalysis
salary.period
  Input should be 'hour', 'day', 'month' or 'year' [type=literal_error, input_value='unknown', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/literal_error
Case ID 9
Attempts 3
First Error 1 validation error for JobPostAnalysis
salary.period
  Input should be 'hour', 'day', 'month' or 'year' [type=literal_error, input_value='unknown', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/literal_error
Case ID 11
Attempts 3
First Error 1 validation error for JobPostAnalysis
salary.period
  Input should be 'hour', 'day', 'month' or 'ye

In [ ]:
print(test_results[6]['job_post'])
print('='*80)
print(test_results[6]['final_reslut']['raw'])

我們正在招募 Product Manager，負責規劃 AI SaaS 產品。
    需要具備需求分析、使用者研究、產品路線圖與跨部門溝通能力。
    工作地點在台北，每週可居家工作兩天。
    要求至少 4 年產品管理經驗，薪資未公開。
    熟悉生成式 AI 產品者加分。
{"job_title": "Product Manager", "seniority": "mid", "work_mode": "hybrid", "location": "台北", "salary": {"min_amount": null, "max_amount": null, "currency": "unknown", "period": "unknown"}, "required_skills": ["需求分析", "使用者研究", "產品路線圖", "跨部門溝通"], "nice_to_have_skills": ["生成式 AI 產品"], "years_of_experience": 4, "job_category": "product_management", "risk_flags": []}


In [ ]:
print(test_results[0]['final_reslut']['raw'])

{
"job_title": "AI Engineer",
"seniority": "mid",
"work_mode": "hybrid",
"location": "台北",
"salary": {
"min_amount": 80000,
"max_amount": 120000,
"currency": "TWD"
},
"required_skills": [
"Python",
"FastAPI",
"LangChain",
"向量資料庫"
],
"nice_to_have_skills": [
"Docker",
"Kubernetes"
],
"years_of_experience": 2,
"job_category": "ai_engineering",
"risk_flags": []
}


In [ ]:
retry_cases = [
    result
    for result in test_results
    if result["attempt_count"] > 1
]

print("Retry case count:", len(retry_cases))

for result in retry_cases:
    print(
        "ID:",
        result["id"],
        "| Type:",
        result["case_type"],
        "| Attempts:",
        result["attempt_count"],
        "| Expected:",
        result["likely_error"]
    )

Retry case count: 0


In [ ]:
CHECK_CASE_IDS = [19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30]

for result in test_results:
    if result["id"] in CHECK_CASE_IDS:
        print("=" * 100)
        print("CASE ID:", result["id"])
        print("LIKELY ERROR:", result["likely_error"])
        print("ATTEMPTS:", result["attempt_count"])
        print("PARSED RESULT:")
        print(result["attempts"][-1]["parsed"])

CASE ID: 19
LIKELY ERROR: years_above_40
ATTEMPTS: 1
PARSED RESULT:
None
CASE ID: 20
LIKELY ERROR: salary_max_below_min
ATTEMPTS: 1
PARSED RESULT:
None
CASE ID: 21
LIKELY ERROR: unsupported_currency
ATTEMPTS: 1
PARSED RESULT:
None
CASE ID: 22
LIKELY ERROR: unsupported_work_mode
ATTEMPTS: 1
PARSED RESULT:
None
CASE ID: 23
LIKELY ERROR: unsupported_seniority
ATTEMPTS: 1
PARSED RESULT:
None
CASE ID: 24
LIKELY ERROR: unsupported_job_category
ATTEMPTS: 1
PARSED RESULT:
None
CASE ID: 25
LIKELY ERROR: non_integer_experience
ATTEMPTS: 1
PARSED RESULT:
None
CASE ID: 26
LIKELY ERROR: non_json_output
ATTEMPTS: 1
PARSED RESULT:
None
CASE ID: 27
LIKELY ERROR: missing_required_field
ATTEMPTS: 1
PARSED RESULT:
None
CASE ID: 28
LIKELY ERROR: extra_field
ATTEMPTS: 1
PARSED RESULT:
None
CASE ID: 29
LIKELY ERROR: invalid_json
ATTEMPTS: 1
PARSED RESULT:
None
CASE ID: 30
LIKELY ERROR: wrong_root_type_or_wrong_boolean_type
ATTEMPTS: 1
PARSED RESULT:
None
